# LiteLLM demo

Talks to the LiteLLM proxy using `LITELLM_API_KEY`, `LITELLM_BASE_URL` and `DEFAULT_MODEL` from `.env`.
Run from the repo root: `uv run jupyter lab` (see [README](../README.md)).
Script version: [`scripts/demo.py`](../scripts/demo.py).

In [ ]:
import os
from dotenv import load_dotenv
import litellm

load_dotenv(".env")

API_KEY = os.environ["LITELLM_API_KEY"]
BASE_URL = os.environ["LITELLM_BASE_URL"]
MODEL = os.environ["DEFAULT_MODEL"]

litellm.api_base = BASE_URL
litellm.api_key = API_KEY

# litellm needs a provider prefix for custom proxy model names
LITELLM_MODEL = f"openai/{MODEL}"

print(f"base_url : {BASE_URL}")
print(f"model    : {MODEL}")

In [ ]:
# List models exposed by the proxy
import httpx

r = httpx.get(
    BASE_URL + "/models",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
r.raise_for_status()
models = r.json()["data"]
print(f"{len(models)} model(s) available:\n")
for m in models:
    print(f"  {m.get('id'):40s}  (owned_by={m.get('owned_by', '?')})")

In [ ]:
# Hello world
import textwrap

resp = litellm.completion(
    model=LITELLM_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise, friendly assistant."},
        {"role": "user", "content": "Write me a 2 paragraph intro to getting started with local AI. What's an iGPU vs dGPU? Pros, cons? MoE vs Dense?"},
    ],
    # max_tokens=120,
)

print(textwrap.fill(resp.choices[0].message.content, width=80))
print(f"\n[usage] {resp.usage.prompt_tokens} prompt / {resp.usage.completion_tokens} completion tokens")